# Investigate Thematic Divergence

This notebook focussed on occurences of semantic divergence between the masked 'machine' token and the 'predictions'. We filtered predictions to only include sentences where the 'machine' is confused with another thematic area. This notebook focusses on the human-machine confusion. 

We there focus in human-type predictions, i.e. occurences where the LMs suggest as human-word as a filler for the machine token. 

For more information on the words we included, inspect `250_freq_pred_KB_edit.txt` and the `1-compute-gradients.ipynb`.


## What can this tell us?

We look at occurences where BERT confused the machine with human words.

We want to understand the divergence, and gauge, which **semantic** and **syntactic** patterns trigger the model towards 'human'-like predictions.

We can look more broadly which words exert push a pull related to (or away from) human or machine agency.

We only look at a smaller subset of examples, based on the existing top 10 predictions recorded in the spreadsheet created by Mariona.

We try to improve and add a few more features:
- a comparative analysis of newspapers and books, i.e. using both newspaper and book data as well as the model trained on these collections
- a more refined syntactic analysis, which includes the syntactic relation of the context token to the masked token

# On Terminology



### A thematic analysis, contrasting two concepts.

We want to understand the semantic intermingling of two distinct concepts, more specifically how the machine are related to, or portrayed as, 'human'-like entities. In this scenario, 'machine' is the TargetMaskedToken (the token that actually appeared in the text) and 'human' is the **predicted contrastive theme**, i.e. a set of thematically related words, predicted by BERT in the position of the masked machine token.

### Target vs. Contrastive Theme

One of the concepts is the **"target concept"**, to other one the **"contrastive concept"**. Each analysis starts with a target concept or token, or `TargetMaskedToken`, this is the starting point of the analysis, for example the word 'machine'. This tokens has two associated datasets. 
    - a TargetMaskedToken maps onto a set of workds that capture the target concept
    - sentences which contain the masked token (`df_target_sent`)
    - contribution obtained by from the integrated gradients method (abbr "ig", `df_target_ig`) 

The `TargetMaskedToken` related to a `ContrastiveTheme`, in this case 'human' like entitites which we filtered from BERT predictions. 

To understand language model predictions, we study both the `TargetMaskedToken` and the `ContrastiveTheme` in two scenarios: **observed** and **counterfactual**. 

 Observed refers to scenarios in which the target of constrastive token appears in the sentences. We than look what tokens contribute to predicting the actual word use. 
 
 In counterfactual scenarios, we measure which words contributed to the ''wrong'' predictions, i.e. what caused BERT to confuse machines with human entities?

## Setting things up and loading the data

In [1]:
import pandas as pd
import json
from tqdm import tqdm
from explain import *
from pathlib import Path

In [2]:
collection,genre_suffix = 'blb',''
if collection == 'blb':
  genre_suffix = '_with_genre'

TargetMaskedToken = 'machine' # the token to be masked in the target sentence

dataPath = 'input_data' # change to '.' when working in colab 
processedFolder = 'gradient_data' # change '.' when working in colab
predCol = "pred_bert_1760_1900"
resultType = 'pred_kw_filtered' # pred | pred_kw_filter

print(f"This analysis focuses on '{TargetMaskedToken}'.")

This analysis focuses on 'machine'.


In [3]:
# load the original sentences with the predicted tokens
df_sent = pd.read_csv(f'{dataPath}/{collection}_{TargetMaskedToken}{genre_suffix}_{resultType}.tsv', index_col=0, sep='\t').reset_index(drop=True)
print(f'We have {df_sent.shape[0]} sentences for the target token {TargetMaskedToken} in the {collection} collection.')
df_ig = pd.read_csv(f'{processedFolder}/results_{collection}_{TargetMaskedToken}_{resultType}_processed.csv', index_col=0 )
print(f'We have {df_ig.shape[0]} explanations for the target token {TargetMaskedToken}.')


We have 19003 sentences for the target token machine in the blb collection.
We have 2173047 explanations for the target token machine.


In [4]:
# check if ids are aligned between gradient data and sentences
print(df_ig['id'].nunique(), df_sent.shape[0])
print(df_ig[df_ig['id'] == 0].Token, df_sent.iloc[0].currentSentence) 

19003 19003
0               "
1            upon
2             the
3          ground
4               -
          ...    
137    fanatacism
138            of
139     crusaders
140             .
141             "
Name: Token, Length: 142, dtype: str " Upon the ground - where they had kneeled, His men would die or win the field, — " and friends and adversaries alike bear testimony, that in his camp, — " the most rigid discipline was found in company with the fiercest resolution; that his troops moved to victory with the precision of machines, while burning with the wildest fanatacism of crusaders. "


In [5]:
len(eval(df_sent.pred_bert_1760_1900.iloc[0]))

20

# Pushing towards Humanity

To find our which contextual element push the prediction to towards "human" type entitites, we start with the most generic type of question, just looking at all the all the words within the sentence across all the different (human-type) predictions.

Let's see which contextual tokens push BLERT to predicting human-like entities. Please note that the analysis below repeats sentencess, we look at all sentences and all the filtered keywords.

In [6]:

targetTokens = ['machine','machines']
df_comparisonConcept = df_ig[
    (~df_ig['Target'].isin(targetTokens)) # we exclude the target token itself, as we are interested in other tokens that are predictive of the contrastive concept
                ].groupby('Token').agg(
                        count=('id', 'count'),identifiers=('id', set),avg_score=('Score', 'mean')
                    ).reset_index()


In [7]:
# please note that this repeats sentences, this acros all sentences with all the filtered keywords
min_count = 10
df_result = df_comparisonConcept[df_comparisonConcept['count'] >= min_count].sort_values(by='avg_score', ascending=False)
df_result.head(10)

,Token,count,identifiers,avg_score
32576,toy,10,"{18283, 3725, 18542, 11440, 8913, 3282, 8469}",0.429404
27104,rides,12,"{17736, 3850, 17681, 4466, 3829, 17053}",0.403356
22789,operators,15,"{6690, 11275, 1039, 11125, 12181, 12119, 15832...",0.399010
18874,lating,13,"{10496, 4354, 3527, 11638, 888}",0.353806
27584,ry,45,"{12034, 2055, 2697, 2059, 13844, 8213, 13207, ...",0.348455
19064,legislative,29,"{13442, 6793, 13325, 15889, 12692, 3605, 3606,...",0.336990
11403,electrical,67,"{18563, 18442, 6036, 16404, 17044, 4377, 10265...",0.334636
28794,shops,2494,"{8204, 8206, 8209, 8212, 8213, 8218, 8220, 822...",0.330775
8769,crusaders,11,"{0, 17569, 17570, 2951, 12200, 16072, 14480, 1...",0.322467
35528,wits,13,"{9179, 10635, 14919, 10671}",0.320851


# Explore Subthemes

Ok, the above is maybe not very helpful. Let's look at subthemes, e.g. 'child' or 'woman' related terms, both together as well as individually?

In [8]:
wordList = ['child','children','boy','girl', 'boys', 'girls','baby','babies']

df_comparisonConcept = df_ig[
    (df_ig['Target'].isin(wordList)) # we exclude the target token itself, as we are interested in other tokens that are predictive of the contrastive concept
                ].groupby('Token').agg(
                        count=('id', 'count'),identifiers=('id', set),avg_score=('Score', 'mean')
                    ).reset_index()


In [9]:

min_count = 10
df_result = df_comparisonConcept[df_comparisonConcept['count'] >= min_count].sort_values(by='avg_score', ascending=False)
df_result.head(10)

,Token,count,identifiers,avg_score
4838,mowing,15,"{13026, 4579, 4580, 10181, 11942, 10024, 17484...",0.360795
4667,mere,354,"{9217, 15367, 15879, 17927, 7179, 3599, 3091, ...",0.315697
4506,man,238,"{12288, 16386, 9220, 5, 4101, 5641, 7179, 9227...",0.311965
2322,dozen,10,"{13410, 5027, 5904, 14577, 17234, 4181, 8028}",0.302689
6583,sewing,214,"{17920, 1, 2, 3, 13314, 18947, 10, 11, 524, 12...",0.298924
6841,soldier,17,"{640, 8257, 5603, 5541, 15879, 4588, 10287, 51...",0.287135
834,bathing,52,"{8833, 10629, 6790, 15110, 3592, 11150, 1424, ...",0.284496
6553,servant,16,"{2305, 14785, 13315, 17506, 4550, 5547, 10475,...",0.281555
1420,child,55,"{12427, 11021, 12431, 10770, 918, 6040, 924, 1...",0.279318
3674,horse,13,"{6593, 6182, 18793, 5900, 13613, 13234, 17012,...",0.274948


In [10]:
pd.set_option('display.max_colwidth', 200)
df_sent.iloc[list(df_result.loc[6841].identifiers)].currentSentence

640                                                                                                                                       The French soldier is not a mere machine in the eyes of his superiors.
8257                            A kiss as well, or damn me if I do not put this young kissing machine beneath my trusty blade, " said a drunken soldier, breaking suddenly in upon the sweetness of their tryst.
5603                                                                                                   The English soldier fights, whde in the army, with all the bravery of the Briton, but it is as a machine.
5541     If we governed, should we take the boy of poor parents, just as he was able to repay them for some of their painful care, and 165 THE STORY OF CHRISTINE ROCHEFORT send him to waste three of his be...
15879                       In this costume, with a suitable instruction in the mystery of marching and countermarching, he is as mere a machine, and, consequently,

# Zoom in on examples

In [11]:
modelName = "Livingwithmachines/bert_1760_1900"
explainer = MaskedLMExplainer(model_name=modelName, device=pick_device())

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: Livingwithmachines/bert_1760_1900
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
idx = 8257
sentence = df_sent.iloc[idx].maskedSentence
targets =list(set(df_ig[df_ig['id'] == idx].Target.unique()).intersection(set(wordList)))

In [13]:
print(f"Sentence: {sentence}")
print(f"Targets: {targets}")

Sentence: A kiss as well , or damn me if I do not put this young kissing [MASK] beneath my trusty blade , " said a drunken soldier , breaking suddenly in upon the sweetness of their tryst .
Targets: ['girl']


In [14]:
# select target
target = targets[0] 

In [15]:

highlight_context_tokens(explainer, sentence, target=target, word_agg="mean")

Explaining:   0%|          | 0/1 [00:00<?, ?it/s]

'\n    <div id="tokviz_734224902c684abeba3459249cc62542">\n      <div style=\'margin-bottom:6px;\'>\n        <b>Target:</b> <code>girl</code>\n      </div>\n      <div style=\'margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;\'>\n        <span style=\'background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;\'>&#9646; predicts</span>\n        <span style=\'background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;\'>&#9646; opposes</span>\n        <span style=\'background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;\'>[target] mask position</span>\n      </div>\n      <div style=\'line-height:2.4; font-size:15px;\'>\n        <span class=\'tok\' data-score=\'0.079899\' style=\'background:rgba(30, 136, 229, 0.272); padding:2px 4px; margin:1px; border-radius:4px; cursor:default;\'>a</span> <span class=\'tok\' data-score=\'0.280614\' style=\'background:rgba(30, 136, 229, 0.705); padding:2px 4px; margin:1p

Rank the sentences by decreasing value of the context word on the predicted word.

In [16]:
target = 'girl'
context_token = 'mowing'
ordered_idx = df_ig[(df_ig['Target']== target) & ((df_ig['Token']==context_token))].sort_values(by='Score', ascending=False)['id'].values

In [17]:
df_sent.iloc[ordered_idx].currentSentence

18268                                                                                                            I felt that in the presence of the gardener and the mowing machine I should be more at my ease.
589                                                                                       — hopping and quarrelling and twittering up and down upon his ladder; may see Jacob and the mowing machine; and — her.
4579                            Phœbe noticed A FAMILIAR TOUCH. 119 that Avice looked whiter than ever as she stood by the mowing machine talking to the old gardener, and ca Ued to her to come into the shade.
10024    And then, just as Miss Sybella was about to respond joyfully, the conference came to an end by a donkey, which was drawing the mowing machine over the lawn in front, beginning to hee haw, and May ...
4580     " We want to hear how you like your busy Use, and to be sure that you are not overworking, " Miss Joy said, by and by, as they sat under the tulip tree, an

In [18]:
sentence = df_sent.iloc[18268].maskedSentence
highlight_context_tokens(explainer, sentence, target=target, word_agg="mean")

Explaining:   0%|          | 0/1 [00:00<?, ?it/s]

'\n    <div id="tokviz_92c520b5918343f49e341d0edb803b95">\n      <div style=\'margin-bottom:6px;\'>\n        <b>Target:</b> <code>girl</code>\n      </div>\n      <div style=\'margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;\'>\n        <span style=\'background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;\'>&#9646; predicts</span>\n        <span style=\'background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;\'>&#9646; opposes</span>\n        <span style=\'background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;\'>[target] mask position</span>\n      </div>\n      <div style=\'line-height:2.4; font-size:15px;\'>\n        <span class=\'tok\' data-score=\'0.065998\' style=\'background:rgba(30, 136, 229, 0.224); padding:2px 4px; margin:1px; border-radius:4px; cursor:default;\'>i</span> <span class=\'tok\' data-score=\'0.083505\' style=\'background:rgba(30, 136, 229, 0.257); padding:2px 4px; margin:1p

## LLM interrogation of a result set

In [47]:
from explain.gpt_annotation import run_gpt_preannotation
OPENAI_API_KEY = ''  # leave blank to use environment variable OPENAI_API_KEY
OPENAI_MODEL = 'gpt-4o'
OPENAI_MAX_WORKERS = 5

COT_SYSTEM_PROMPT = '''You are an expert historian reading excerpts from 19th-century British books.
We want to explore the idea that nineteenth‑century discourse often treats humans, machines, animals, and slaves as interchangeable labouring entities within a shared conceptual field.

Analyse the sentences, and assess to what extent they portray machine as being "alive" or "human-like". 
Do the sentences express anthropomorphism (machines treated as alive) or technomorphism (humans described as machines)?
Consider the context, the actions described, and any anthropomorphic or technomorphic language used. 

Think step-by-step, then output ONLY a JSON object in this exact format (no markdown, no prose, no backticks):
{"reasoning": "<chain-of-thought reasoning>", "label": "anthropomorphic", "technomorphic", or "none"}
'''


In [48]:
df_sample = pd.DataFrame(df_sent.iloc[list(df_result.loc[6841].identifiers)].currentSentence)
df_sample.rename(columns={'currentSentence':'sentence'}, inplace=True)

df_sample.reset_index(drop=True)
df_sample['label'] = ''
df_sample['reasoning'] = ''
df_sample.head(10)

,sentence,label,reasoning
640,The French soldier is not a mere machine in the eyes of his superiors.,,
8257,"A kiss as well, or damn me if I do not put this young kissing machine beneath my trusty blade, "" said a drunken soldier, breaking suddenly in upon the sweetness of their tryst.",,
5603,"The English soldier fights, whde in the army, with all the bravery of the Briton, but it is as a machine.",,
5541,"If we governed, should we take the boy of poor parents, just as he was able to repay them for some of their painful care, and 165 THE STORY OF CHRISTINE ROCHEFORT send him to waste three of his be...",,
15879,"In this costume, with a suitable instruction in the mystery of marching and countermarching, he is as mere a machine, and, consequently, as good a soldier, as can well be imagined.",,
4588,"On one occasion, when at drill, Gould called out to the regiment — "" Steady, gentle men, steady; a soldier is a mere machine.",,
10287,"The soldier boys resorted to all sorts of expedients to "" beat the machine. """,,
5168,"The instruc tion imparted at Hythe is chiefly designed to heighten the soldier 's intelligence, for, OAving to the introduction of the rifle, he is bound hence forth to regard himself as an indivi...",,
12207,"The Russian soldier is a mere machine, without animation or enterprise, and, with the exception of the Kosacks, totally unfit to act as light troops.",,
9783,"The soldier was re garded as a sort of mechanical contrivance, capable of the most daring conduct, and of rendering the most valuable service as long as his officers were present to lead and direc...",,


In [49]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from explain.gpt_annotation import run_gpt_preannotation

summary = run_gpt_preannotation(
    df_sample,
    system_prompt=COT_SYSTEM_PROMPT,
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    n=0,
    max_workers=OPENAI_MAX_WORKERS,
)

Sending 13 snippets to gpt-4o …
  13/13 done  (ok=13, errors=0)

Pre-annotation complete: {'sent': 13, 'ok': 13, 'errors': 0}


In [51]:
df_sample

,sentence,label,reasoning,gpt_error
640,The French soldier is not a mere machine in the eyes of his superiors.,technomorphic,"The sentence suggests that the French soldier is not viewed as a mere machine, implying that there is a tendency to view soldiers as machines. This reflects technomorphism, where humans (soldiers)...",
8257,"A kiss as well, or damn me if I do not put this young kissing machine beneath my trusty blade, "" said a drunken soldier, breaking suddenly in upon the sweetness of their tryst.",technomorphic,"The phrase 'young kissing machine' refers to a person, likely implying that the person is performing the action of kissing in a mechanical or repetitive manner. This is an example of technomorphis...",
5603,"The English soldier fights, whde in the army, with all the bravery of the Briton, but it is as a machine.",technomorphic,"The sentence describes the English soldier as fighting with bravery but qualifies this by saying it is 'as a machine.' This implies that the soldier's actions are mechanical, lacking individual ag...",
5541,"If we governed, should we take the boy of poor parents, just as he was able to repay them for some of their painful care, and 165 THE STORY OF CHRISTINE ROCHEFORT send him to waste three of his be...",technomorphic,"The sentence describes a boy being trained to become 'a mere machine of a soldier.' This implies that the boy, a human, is being described in terms of a machine, suggesting a transformation into s...",
15879,"In this costume, with a suitable instruction in the mystery of marching and countermarching, he is as mere a machine, and, consequently, as good a soldier, as can well be imagined.",technomorphic,"The sentence describes a soldier as being 'as mere a machine' when trained in marching and countermarching. This comparison suggests that the soldier, a human, is being described in terms of a mac...",
4588,"On one occasion, when at drill, Gould called out to the regiment — "" Steady, gentle men, steady; a soldier is a mere machine.",technomorphic,"The sentence explicitly states that 'a soldier is a mere machine,' which is a clear example of technomorphism. It describes humans, specifically soldiers, as machines, emphasizing their role as la...",
10287,"The soldier boys resorted to all sorts of expedients to "" beat the machine. """,none,"The phrase 'beat the machine' suggests a competitive or adversarial relationship between humans and machines. However, it does not explicitly describe the machine as being alive or human-like, nor...",
5168,"The instruc tion imparted at Hythe is chiefly designed to heighten the soldier 's intelligence, for, OAving to the introduction of the rifle, he is bound hence forth to regard himself as an indivi...",technomorphic,The sentence contrasts the soldier's previous perception as a machine with the new expectation to see himself as an individual due to the introduction of the rifle. This implies that soldiers were...,
12207,"The Russian soldier is a mere machine, without animation or enterprise, and, with the exception of the Kosacks, totally unfit to act as light troops.",technomorphic,"The sentence describes the Russian soldier as a 'mere machine,' indicating a lack of animation or enterprise. This comparison suggests that the soldier is being described in terms of a machine, em...",
9783,"The soldier was re garded as a sort of mechanical contrivance, capable of the most daring conduct, and of rendering the most valuable service as long as his officers were present to lead and direc...",technomorphic,"The snippet describes a soldier as a 'mechanical contrivance,' suggesting that the soldier is viewed as a machine that requires external direction to function. This is a clear example of technomor...",


In [64]:
from explain.gpt_annotation import run_gpt_preannotation
OPENAI_API_KEY = ''  # leave blank to use environment variable OPENAI_API_KEY
OPENAI_MODEL = 'gpt-4o'
OPENAI_MAX_WORKERS = 5

COT_SYSTEM_PROMPT = '''You are an average person living and reading in the 19th century.

Analyse the sentences, focuss on the [MASKED] token. What word would you expect to be in the [MASKED] position, based on the context of the sentence?

Think step-by-step, then output ONLY a JSON object in this exact format (no markdown, no prose, no backticks):
{"reasoning": "<chain-of-thought reasoning>", "label": "<expected_word>"}
'''


In [65]:
df_sample = pd.DataFrame(df_sent.iloc[list(df_result.loc[6841].identifiers)].maskedSentence)
df_sample.rename(columns={'maskedSentence':'sentence'}, inplace=True)

df_sample.reset_index(drop=True)
df_sample['label'] = ''
df_sample['reasoning'] = ''

In [66]:
from explain.gpt_annotation import run_gpt_preannotation

summary = run_gpt_preannotation(
    df_sample,
    system_prompt=COT_SYSTEM_PROMPT,
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    n=0,
    max_workers=OPENAI_MAX_WORKERS,
)

Sending 13 snippets to gpt-4o …
  13/13 done  (ok=13, errors=0)

Pre-annotation complete: {'sent': 13, 'ok': 13, 'errors': 0}


In [67]:
df_sample

,sentence,label,reasoning,gpt_error
640,The French soldier is not a mere [MASK] in the eyes of his superiors .,pawn,The sentence is discussing the perception of a French soldier by his superiors. The phrase 'not a mere' suggests that the word following it should be something that implies a lower status or a sim...,
8257,"A kiss as well , or damn me if I do not put this young kissing [MASK] beneath my trusty blade , "" said a drunken soldier , breaking suddenly in upon the sweetness of their tryst .",couple,The sentence describes a drunken soldier interrupting a romantic moment and threatening to use his blade. The phrase 'young kissing [MASK]' suggests that the masked word is a noun describing a per...,
5603,"The English soldier fights , whde in the army , with all the bravery of the Briton , but it is as a [MASK] .",lion,"The sentence is discussing the English soldier's bravery while in the army. The phrase 'it is as a' suggests a comparison or metaphor. In the 19th century, soldiers were often compared to lions fo...",
5541,"If we governed , should we take the boy of poor parents , just as he was able to repay them for some of their painful care , and 165 THE STORY OF CHRISTINE ROCHEFORT send him to waste three of his...",drudge,The sentence discusses the idea of taking a boy from poor parents and sending him to learn something that might not be very beneficial. The phrase 'mere [MASK] of a soldier' suggests that the word...,
15879,"In this costume , with a suitable instruction in the mystery of marching and countermarching , he is as mere a [MASK] , and , consequently , as good a soldier , as can well be imagined .",puppet,"The sentence describes someone in a costume who has been instructed in marching and countermarching, suggesting they are being compared to something or someone that is not genuinely a soldier but ...",
4588,"On one occasion , when at drill , Gould called out to the regiment — "" Steady , gentle men , steady ; a soldier is a mere [MASK] .",machine,The sentence is about a drill and the speaker is addressing soldiers. The phrase 'a soldier is a mere' suggests that the speaker is about to describe a soldier in a simplistic or fundamental way. ...,
10287,"The soldier boys resorted to all sorts of expedients to "" beat the [MASK] . """,drum,"The phrase 'beat the [MASK]' is often used in the context of overcoming or outsmarting something. In the context of soldiers, this phrase is commonly associated with 'beat the system' or 'beat the...",
5168,"The instruc tion imparted at Hythe is chiefly designed to heighten the soldier 's intelligence , for , OAving to the introduction of the rifle , he is bound hence forth to regard himself as an ind...",unit,"The sentence discusses the change in a soldier's role due to the introduction of the rifle. It suggests that previously, soldiers were not seen as individuals but rather as part of a collective or...",
12207,"The Russian soldier is a mere [MASK] , without animation or enterprise , and , with the exception of the Kosacks , totally unfit to act as light troops .",automaton,"The sentence describes the Russian soldier as lacking animation or enterprise, suggesting a lack of initiative or individuality. The word 'mere' implies something insignificant or basic. Given the...",
9783,"The soldier was re garded as a sort of mechanical contrivance , capable of the most daring conduct , and of rendering the most valuable service as long as his officers were present to lead and dir...",soldier,"The sentence describes a soldier as a mechanical contrivance that requires the presence and supervision of officers to function effectively. The context suggests that without this supervision, the...",


## Temporal Filtering

In [ ]:
indices = df_sent[df_sent['date'].between(1850, 1900)].index

In [ ]:
targetTokens = ['machine','machines']
df_comparisonConcept = df_ig[
    (~df_ig['Target'].isin(targetTokens)) & \
    (df_ig['id'].isin(indices))
                ].groupby('Token').agg(
                        count=('id', 'count'),identifiers=('id', set),avg_score=('Score', 'mean')
                    ).reset_index()


In [ ]:
min_count = 50
df_comparisonConcept[df_comparisonConcept['count'] >= min_count].sort_values(by='avg_score', ascending=False).head(10)

In [ ]:
df_ig_temporal = df_ig.merge(df_sent[['date']], left_on='id', right_index=True, how='left').fillna(-1)
df_ig_temporal['decade'] = (df_ig_temporal['date']//10)*10
df_ig_temporal.head()

In [ ]:
by_year = df_ig_temporal[
                         (~df_ig_temporal['Target'].isin(targetTokens))
                         ].groupby(['date','Token']).agg(
                                count=('id', 'count'),sum_score=('Score', 'sum')
                            ).reset_index()


In [ ]:
by_year[(by_year.date.between(1800, 1900)) & (by_year.Token.isin(['child','children']))].groupby('decade').agg(
    count=('count', 'sum'),avg_score=('sum_score', 'mean')
).reset_index().plot(x='decade', y='avg_score', kind='line', title='Average score for token "child" over decades')


In [ ]:
by_year[by_year.date > 1800].plot(x='date', y='avg_score', kind='line', title='Average score for token "child" over years')

In [ ]:
by_decade = df_ig_temporal[df_ig_temporal.Token.isin(['child','children'])].groupby('decade').agg(
    count=('id', 'count'),avg_score=('Score', 'mean')).reset_index()
by_decade[by_decade.decade > 1800].plot(x='decade', y='avg_score', kind='line', title='Average score for token "child" over decades')

In [ ]:
by_decade[by_decade.decade > 1800].plot(x='decade', y='count', kind='line', title='Average score for token "child" over decades')

In [ ]:

comparisonTokens = ['child','children']


df_comparisonConcept = df_by_token[& \
         (df_by_token['count'] >= min_count )
         ]

In [ ]:
df_comparisonConcept.head(10)

In [ ]:

target = by_token[(by_token['Target_simplified'].isin(target_tokens)) & \
         (by_token['count'] >= min_count )
         ]
print(comparison.shape[0], target.shape[0])

In [ ]:
df_ig.columns

In [ ]:
df_ig.Target.unique()

In [ ]:
display(
    df_ig[(df_ig.id==9)               
                 ][["id", "Target", "Token",'Score', "Score_normalized", "mask_syntax_relation", "mask_constituent_relation"]]
)

In [ ]:
df

In [ ]:
df_sent.pred_bert_1760_1900.iloc[0]

In [ ]:
df_ig['Target_position'].unique()